# 01 构建 ViViT 对齐历史索引

本 Notebook 复用 `ViT/data/feature_store`，只生成 `[样本数,20]` 的历史行号矩阵；年度 train、validation、prediction 在运行时直接读取 ViT 原始 split，不复制特征或 split 文件。


## 步骤 1：定位目录并导入公共实现

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

VIVIT_DIR = Path.cwd().resolve()
if VIVIT_DIR.name != "ViViT":
    VIVIT_DIR = VIVIT_DIR / "ViViT"
if not (VIVIT_DIR / "stage01_sequence_data.py").exists():
    raise FileNotFoundError("无法定位 ViViT/stage01_sequence_data.py")
sys.path.insert(0, str(VIVIT_DIR))

from stage01_sequence_data import (
    HISTORY_INDICES_PATH,
    SEQUENCE_MANIFEST_PATH,
    SEQUENCE_SUMMARY_PATH,
    audit_split_alignment,
    build_sequence_store,
)

print(f"ViViT 目录：{VIVIT_DIR}")
print(f"feature_store：{(VIVIT_DIR / 'data' / 'feature_store').resolve()}")
print(f"aligned store：{HISTORY_INDICES_PATH.parent}")


## 步骤 2：生成固定 20 日槽位

每个位置只匹配同股票的精确 `target_date_id-lag`，不前向填充、不使用未来数据。缺失位置记为 `-1`；Collator 将对应 17 通道图像补零，并生成 `[B,20]` 布尔 `frame_mask`。目标日不因历史缺口被过滤，也不再每 5 日降采样；2025 prediction 与 ViT 一致为空。


In [ ]:
# 若源 ViT manifest 改变，必须显式设为 True 重建，禁止静默复用。
OVERWRITE_ALIGNED_STORE = False
sequence_manifest = build_sequence_store(overwrite=OVERWRITE_ALIGNED_STORE)
sequence_manifest


## 步骤 3：审计 aligned store 与 ViT split


In [ ]:
if not HISTORY_INDICES_PATH.exists() or not SEQUENCE_MANIFEST_PATH.exists():
    raise FileNotFoundError("aligned sequence store 没有完整生成")

sequence_summary = pd.read_parquet(SEQUENCE_SUMMARY_PATH)
if sequence_summary["forecast_year"].tolist() != list(range(2018, 2026)):
    raise AssertionError("预测年度必须完整覆盖 2018～2025")
if sequence_summary.loc[sequence_summary.forecast_year == 2025, "prediction_sample_count"].item() != 0:
    raise AssertionError("2025 prediction 必须与 ViT 一致为空")

split_audit = audit_split_alignment()
if not split_audit["array_equal"].all():
    raise AssertionError("ViViT 与 ViT split 不一致")
display(sequence_summary)
display(split_audit)
